In [ ]:
import os
import multiprocessing
import numpy as np
import pandas as pd
import joblib
import json
import copy
from functools import partial

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau

import optuna
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, classification_report, confusion_matrix

import warnings
warnings.filterwarnings("ignore")

In [ ]:
os.environ['OPENBLAS_NUM_THREADS'] = '8'
os.environ['MKL_NUM_THREADS'] = '8'
os.environ['OMP_NUM_THREADS'] = '8'

gpu_id = 3
device = torch.device(f"cuda:{gpu_id}" if torch.cuda.is_available() else "cpu")
torch.cuda.set_device(gpu_id)
torch.backends.cudnn.benchmark = True
print(f"Dispositivo: {device}")

In [ ]:
X = pd.read_pickle("Sets_Xy/X.pkl")
y = pd.read_pickle("Sets_Xy/y.pkl")

from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42)

le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_val_encoded = le.transform(y_val)
y_test_encoded = le.transform(y_test)

class_names = le.classes_
num_classes = len(class_names)

X_train_full = joblib.load("Sets_Oversampling_2/X_train_none.joblib")
if isinstance(X_train_full, pd.DataFrame):
    X_train_full = X_train_full.values
X_train_full = X_train_full.astype(np.float32)

y_train_full = joblib.load("Sets_Oversampling_2/y_train_none.joblib")
if isinstance(y_train_full, (pd.DataFrame, pd.Series)):
    y_train_full = y_train_full.values.ravel()
y_train_full = y_train_full.astype(np.int64)

X_val = pd.read_pickle("Sets_Post_Scaled_Imp/X_val_scaled_bayesian.pkl")
if isinstance(X_val, pd.DataFrame):
    X_val = X_val.values
X_val = X_val.astype(np.float32)

X_test = pd.read_pickle("Sets_Post_Scaled_Imp/X_test_scaled_bayesian.pkl")
if isinstance(X_test, pd.DataFrame):
    X_test = X_test.values
X_test = X_test.astype(np.float32)

y_val_1d = np.array(y_val_encoded)
y_test_1d = np.array(y_test_encoded)

print(f"Datos listos. Clases: {num_classes}, Features: {X_train_full.shape[1]}")
print(f"Train: {X_train_full.shape[0]}, Val: {X_val.shape[0]}, Test: {X_test.shape[0]}")

le = LabelEncoder()
le.fit(pd.read_pickle("Sets_Xy/y.pkl"))
class_names = le.classes_
num_classes = len(class_names)

print(f"Datos listos. Clases: {num_classes}, Features: {X_train_full.shape[1]}")

In [ ]:
clases_por_experto = {
    1: [0, 1, 12, 13, 14],
    2: [2, 4, 10],
    3: [3, 5, 6],
    4: [7, 11],
    5: [8, 9]
}

class FastTensorDataLoader:
    def __init__(self, X, y, batch_size, shuffle=False):
        self.X = torch.from_numpy(np.asarray(X, dtype=np.float32))
        self.y = torch.from_numpy(np.asarray(y, dtype=np.int64))
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.n = len(X)

    def __iter__(self):
        if self.shuffle:
            idx = torch.randperm(self.n)
            self.X = self.X[idx]
            self.y = self.y[idx]
        self.i = 0
        return self

    def __next__(self):
        if self.i >= self.n:
            raise StopIteration
        end = min(self.i + self.batch_size, self.n)
        batch_X = self.X[self.i:end].to(device, non_blocking=True)
        batch_y = self.y[self.i:end].to(device, non_blocking=True)
        self.i = end
        return batch_X, batch_y

    def __len__(self):
        return (self.n + self.batch_size - 1) // self.batch_size

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, reduction='mean'):
        super().__init__()
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        return focal_loss

In [ ]:
class TabTransformerFast(nn.Module):
    def __init__(self, input_dim, num_classes, d_model=64, nhead=4,
                 num_layers=2, num_tokens=16, dim_feedforward=128, dropout=0.1):
        super().__init__()
        
        # Validar compatibilidad
        if d_model % nhead != 0:
            nhead = 2 if d_model % 2 == 0 else 1
        
        self.num_tokens = num_tokens
        self.d_model = d_model

        # Proyección inicial
        self.feature_projection = nn.Linear(input_dim, num_tokens * d_model)

        # CLS token aprendible
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        
        # Codificación posicional simplificada
        self.register_buffer('pos_encoding', 
                           torch.randn(1, num_tokens + 1, d_model) * 0.02)

        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation='gelu',
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # Head de clasificación
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, num_classes)
        )

        # Inicialización de pesos
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.xavier_uniform_(module.weight)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)

    def forward(self, x):
        batch_size = x.shape[0]
        
        # Proyectar a tokens
        x = self.feature_projection(x)
        x = x.view(batch_size, self.num_tokens, self.d_model)

        # Agregar CLS token
        cls = self.cls_token.expand(batch_size, -1, -1)
        x = torch.cat([cls, x], dim=1)
        
        # Agregar posición y transformer
        x = x + self.pos_encoding
        x = self.transformer(x)
        
        # Clasificar usando CLS token
        x = self.norm(x[:, 0, :])
        return self.head(x)

In [ ]:
class DynamicGatingNetwork(nn.Module):
    def __init__(self, input_dim, num_experts, hidden_layers, dropout_rate, activation_name):
        super().__init__()
        
        activations = {
            'leaky_relu': nn.LeakyReLU(0.01),
            'gelu': nn.GELU(),
            'relu': nn.ReLU()
        }
        act = activations.get(activation_name, nn.GELU())
        
        layers = []
        in_f = input_dim
        for out_f in hidden_layers:
            layers.extend([
                nn.Linear(in_f, out_f),
                nn.BatchNorm1d(out_f),
                act,
                nn.Dropout(dropout_rate)
            ])
            in_f = out_f
        
        layers.append(nn.Linear(in_f, num_experts))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return torch.softmax(self.net(x), dim=1)

In [ ]:

class MoESoftRouting(nn.Module):
    def __init__(self, lista_expertos, gating_network):
        super().__init__()
        self.experts = nn.ModuleList(lista_expertos)
        self.gating = gating_network
        
        for expert in self.experts:
            for param in expert.parameters():
                param.requires_grad = True

    def forward(self, x):
        weights = self.gating(x).unsqueeze(-1)
        expert_outputs = torch.stack([expert(x) for expert in self.experts], dim=1)
        return torch.sum(expert_outputs * weights, dim=1)

In [ ]:

def objective_experto_tabtransformer(trial, X_train, y_train, X_val, y_val, input_dim, nombre_experto, clases_objetivo):
    d_model = trial.suggest_categorical('d_model', [64, 128])
    num_tokens = trial.suggest_categorical('num_tokens', [8, 16])
    nhead = trial.suggest_categorical('nhead', [2, 4])
    
    if d_model % nhead != 0:
        nhead = 2
    
    num_layers = trial.suggest_int('num_layers', 1, 2)
    dim_feedforward = trial.suggest_categorical('dim_feedforward', [128, 256])
    dropout_rate = trial.suggest_float('dropout_rate', 0.1, 0.3)
    
    lr = trial.suggest_float('lr', 5e-4, 3e-3, log=True)
    weight_decay = trial.suggest_float('weight_decay', 1e-5, 1e-3, log=True)
    focal_gamma = trial.suggest_float('focal_gamma', 0.0, 2.0)
    batch_size = trial.suggest_categorical('batch_size', [1024, 2048])
    
    # DataLoaders
    train_loader = FastTensorDataLoader(X_train, y_train, batch_size, shuffle=True)
    val_loader = FastTensorDataLoader(X_val, y_val, batch_size * 2, shuffle=False)
    
    # Modelo
    model = TabTransformerFast(
        input_dim=input_dim,
        num_classes=num_classes,
        d_model=d_model,
        nhead=nhead,
        num_layers=num_layers,
        num_tokens=num_tokens,
        dim_feedforward=dim_feedforward,
        dropout=dropout_rate
    ).to(device)
    
    # Entrenamiento
    criterion = FocalLoss(gamma=focal_gamma)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5)
    scaler = torch.amp.GradScaler(device.type)
    
    best_f1 = 0.0
    patience_counter = 0
    best_state = None
    
    for epoch in range(100):
        # Entrenamiento
        model.train()
        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()
            
            with torch.amp.autocast(device.type):
                outputs = model(batch_X)
                loss = criterion(outputs, batch_y)
            
            scaler.scale(loss).backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
        
        # Validación
        model.eval()
        y_true_all, y_pred_all = [], []
        
        with torch.no_grad():
            for batch_X, batch_y in val_loader:
                with torch.amp.autocast(device.type):
                    outputs = model(batch_X)
                y_true_all.append(batch_y.cpu())
                y_pred_all.append(outputs.argmax(1).cpu())
        
        y_true_all = torch.cat(y_true_all).numpy()
        y_pred_all = torch.cat(y_pred_all).numpy()
        
        current_f1 = f1_score(y_true_all, y_pred_all,
                             labels=clases_objetivo,
                             average='macro',
                             zero_division=0)
        
        scheduler.step(current_f1)
        
        trial.report(current_f1, epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()
        
        if current_f1 > best_f1:
            best_f1 = current_f1
            patience_counter = 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            patience_counter += 1
        
        if patience_counter >= 8:
            break
    
    # Guardar mejor modelo
    os.makedirs("Models_Experts_Third_Architecture", exist_ok=True)
    ruta = f"Models_Experts_Third_Architecture/{nombre_experto}_trial_{trial.number}.pth"
    torch.save(best_state, ruta)
    
    trial.set_user_attr("ruta_pesos", ruta)
    trial.set_user_attr("best_params", {
        "d_model": d_model,
        "nhead": nhead,
        "num_layers": num_layers,
        "num_tokens": num_tokens,
        "dim_feedforward": dim_feedforward,
        "dropout_rate": dropout_rate
    })
    
    del model, optimizer, scheduler, scaler, train_loader, val_loader
    torch.cuda.empty_cache()
    
    return best_f1

In [ ]:

def objective_moe_gating_tabtransformer(trial, X_train, y_train, X_val, y_val,input_dim, lista_expertos, oversampling):

    batch_size = trial.suggest_categorical('batch_size', [1024, 2048])
    dropout_rate = trial.suggest_float('dropout_rate', 0.1, 0.3)
    activation_name = trial.suggest_categorical('activation', ['leaky_relu', 'gelu'])
    
    lr_gating = trial.suggest_float('lr_gating', 1e-4, 5e-3, log=True)
    lr_experts = trial.suggest_float('lr_experts', 1e-6, 1e-4, log=True)
    weight_decay = trial.suggest_float('weight_decay', 1e-6, 1e-3, log=True)
    focal_gamma = trial.suggest_float('focal_gamma', 0.0, 2.0)
    
    n_layers = trial.suggest_int('n_layers', 1, 2)
    hidden_layers = []
    base_units = trial.suggest_int('n_units_l0', 128, 512, step=128)
    hidden_layers.append(base_units)
    
    prev_units = base_units
    for i in range(1, n_layers):
        units = trial.suggest_int(f'n_units_l{i}', 64, prev_units, step=64)
        hidden_layers.append(units)
        prev_units = units
    
    # DataLoaders
    train_loader = FastTensorDataLoader(X_train, y_train, batch_size, shuffle=True)
    val_loader = FastTensorDataLoader(X_val, y_val, batch_size * 2, shuffle=False)
    
    # Modelo
    gating = DynamicGatingNetwork(input_dim, len(lista_expertos),
                                  hidden_layers, dropout_rate, activation_name)
    moe = MoESoftRouting(lista_expertos, gating).to(device)
    
    # Entrenamiento
    criterion = FocalLoss(gamma=focal_gamma)
    optimizer = optim.AdamW([
        {'params': moe.gating.parameters(), 'lr': lr_gating},
        {'params': moe.experts.parameters(), 'lr': lr_experts}
    ], weight_decay=weight_decay)
    
    scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)
    scaler = torch.amp.GradScaler(device.type)
    
    best_f1 = 0.0
    patience_counter = 0
    best_state = None
    
    for epoch in range(100):
        # Entrenamiento
        moe.train()
        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()
            
            with torch.amp.autocast(device.type):
                outputs = moe(batch_X)
                loss = criterion(outputs, batch_y)
            
            scaler.scale(loss).backward()
            torch.nn.utils.clip_grad_norm_(moe.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
        
        # Validación
        moe.eval()
        y_true_all, y_pred_all = [], []
        
        with torch.no_grad():
            for batch_X, batch_y in val_loader:
                with torch.amp.autocast(device.type):
                    outputs = moe(batch_X)
                y_true_all.append(batch_y.cpu())
                y_pred_all.append(outputs.argmax(1).cpu())
        
        y_true_all = torch.cat(y_true_all).numpy()
        y_pred_all = torch.cat(y_pred_all).numpy()
        
        current_f1 = f1_score(y_true_all, y_pred_all,
                             average='macro',
                             zero_division=0)
        
        scheduler.step(current_f1)
        
        trial.report(current_f1, epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()
        
        if current_f1 > best_f1:
            best_f1 = current_f1
            patience_counter = 0
            best_state = copy.deepcopy(moe.state_dict())
        else:
            patience_counter += 1
        
        if patience_counter >= 6:
            break
    
    # Guardar
    os.makedirs("Models_Gating_Third_Architecture", exist_ok=True)
    ruta = f"Models_Gating_Third_Architecture/Gating_{oversampling}_trial_{trial.number}.pth"
    torch.save(best_state, ruta)
    
    trial.set_user_attr("ruta_pesos", ruta)
    trial.set_user_attr("hidden_layers", hidden_layers)
    
    del moe, gating, optimizer, scheduler, scaler, train_loader, val_loader
    torch.cuda.empty_cache()
    
    return best_f1

In [ ]:
def automatizar_fase1_expertos(metodo_oversampling, X_val, y_val_1d,clases_por_experto, n_trials=100):

    lista_expertos = []
    os.makedirs("Models_Experts_Third_Architecture", exist_ok=True)
    os.makedirs("Models_Final_Third_Architecture", exist_ok=True)
    
    for i in range(1, 6):
        print(f"  Entrenando Experto {i}")
        
        # Cargar datos
        X_exp = joblib.load(
            f"Sets_Oversampling_MoE_2/X_train_{metodo_oversampling}_Experto_{i}.joblib"
        ).astype(np.float32)
        if isinstance(X_exp, pd.DataFrame):
            X_exp = X_exp.values
            
        y_exp = joblib.load(
            f"Sets_Oversampling_MoE_2/y_train_{metodo_oversampling}_Experto_{i}.joblib"
        )
        if isinstance(y_exp, (pd.DataFrame, pd.Series)):
            y_exp = y_exp.values.ravel()
        y_exp = y_exp.astype(np.int64)
        
        input_dim = X_exp.shape[1]
        nombre_exp = f"Experto_{i}_{metodo_oversampling}"
        
        # Optuna study
        obj_func = partial(
            objective_experto_tabtransformer,
            X_train=X_exp, y_train=y_exp,
            X_val=X_val, y_val=y_val_1d,
            input_dim=input_dim,
            nombre_experto=nombre_exp,
            clases_objetivo=clases_por_experto[i]
        )
        
        study = optuna.create_study(
            direction="maximize",
            study_name=nombre_exp,
            pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=10)
        )
        study.optimize(obj_func, n_trials=n_trials, n_jobs=1)
        
        best_params = study.best_trial.user_attrs['best_params']
        ruta_pesos = study.best_trial.user_attrs['ruta_pesos']
        
        experto = TabTransformerFast(
            input_dim=input_dim,
            num_classes=num_classes,
            d_model=best_params['d_model'],
            nhead=best_params['nhead'],
            num_layers=best_params['num_layers'],
            num_tokens=best_params['num_tokens'],
            dim_feedforward=best_params['dim_feedforward'],
            dropout=best_params['dropout_rate']
        ).to(device)
                
        experto.load_state_dict(torch.load(ruta_pesos, map_location=device))
        lista_expertos.append(experto)
        
        # Guardar configuración
        with open(f"Models_Final_Third_Architecture/Arquitectura_{nombre_exp}.json", "w") as f:
            json.dump(best_params, f, indent=4)
        
        print(f"Best F1: {study.best_value:.4f}")
        
        del X_exp, y_exp
        torch.cuda.empty_cache()
    
    return lista_expertos

In [ ]:

def automatizar_fase2_gating(metodo_oversampling, lista_expertos,X_train, y_train, X_val, y_val,input_dim, n_trials=100):
    
    print(f"Entrenando red de enrutamiento...")
    
    obj_func = partial(
        objective_moe_gating_tabtransformer,
        X_train=X_train, y_train=y_train,
        X_val=X_val, y_val=y_val,
        input_dim=input_dim,
        lista_expertos=lista_expertos,
        oversampling=metodo_oversampling
    )
    
    study = optuna.create_study(
        direction="maximize",
        study_name=f"Gating_TabTransformer_{metodo_oversampling}",
        pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=10)
    )
    study.optimize(obj_func, n_trials=n_trials, n_jobs=1)
    
    # Cargar mejor modelo
    best_trial = study.best_trial
    best_params = best_trial.params
    hidden_layers = best_trial.user_attrs['hidden_layers']
    
    # Guardar configuración completa
    config = {
        "input_dim": int(input_dim),
        "num_experts": len(lista_expertos),
        "hidden_layers": hidden_layers,
        "dropout_rate": float(best_params['dropout_rate']),
        "activation_name": best_params['activation'],
        "lr_gating": float(best_params['lr_gating']),
        "lr_experts": float(best_params['lr_experts']),
        "focal_gamma": float(best_params['focal_gamma'])
    }
    
    os.makedirs("Models_Final_Third_Architecture", exist_ok=True)
    with open(f"Models_Final_Third_Architecture/MoE_{metodo_oversampling}_Arquitectura.json", "w") as f:
        json.dump(config, f, indent=4)
    
    # Crear modelo final
    gating = DynamicGatingNetwork(
        input_dim=input_dim,
        num_experts=len(lista_expertos),
        hidden_layers=hidden_layers,
        dropout_rate=best_params['dropout_rate'],
        activation_name=best_params['activation']
    )
    
    moe_final = MoESoftRouting(lista_expertos, gating).to(device)
    moe_final.load_state_dict(torch.load(best_trial.user_attrs['ruta_pesos'],map_location=device))
    
    print(f"Best F1: {study.best_value:.4f}")
    
    return moe_final

In [ ]:
def save_confusion_matrix(y_true, y_pred, name, phase="Val"):
    os.makedirs("Logs_MoE_Third_Architecture", exist_ok=True)
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(12, 10))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Purples',
                xticklabels=class_names, yticklabels=class_names)
    plt.title(f'MoE TabTransformer - Matriz de Confusión ({phase})')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.savefig(f'Logs_MoE_Third_Architecture/{name}.png', dpi=100)
    plt.close()

In [ ]:

def save_expert_usage_matrix(moe_model, test_loader, method_name):
    moe_model.eval()
    
    y_true_test = []
    all_gating_weights = []
    
    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            with torch.amp.autocast(device.type):
                _ = moe_model(batch_X)
                gating_logits = moe_model.gating.net(batch_X)
                gating_probs = torch.softmax(gating_logits, dim=1)
            
            y_true_test.append(batch_y.cpu().numpy())
            all_gating_weights.append(gating_probs.cpu().numpy())
    
    all_gating_weights = np.vstack(all_gating_weights)
    y_true_test = np.concatenate(y_true_test)
    
    num_experts = all_gating_weights.shape[1]
    expert_usage = np.zeros((num_classes, num_experts))
    
    for i in range(num_classes):
        idx = (y_true_test == i)
        if np.any(idx):
            expert_usage[i] = np.mean(all_gating_weights[idx], axis=0)
    
    plt.figure(figsize=(12, 8))
    sns.heatmap(
        expert_usage,
        annot=True,
        cmap="Blues",
        fmt=".3f",
        xticklabels=[f"Experto {i+1}" for i in range(num_experts)],
        yticklabels=[str(c) for c in class_names]
    )
    plt.title(f'Matriz de Utilización de Expertos - {method_name}')
    plt.ylabel('Clase Real')
    plt.xlabel('Experto')
    plt.tight_layout()
    plt.savefig(f'Logs_MoE_Third_Architecture/Matriz_Utilizacion_{method_name}.png',dpi=100, bbox_inches='tight')
    plt.close()
    
    np.save(f'Logs_MoE_Third_Architecture/Expert_Usage_{method_name}.npy', expert_usage)
    
    return expert_usage

In [ ]:
def automatizar_evaluacion_moe(moe_model, X_val, y_val, X_test, y_test, class_names, method_name):
    
    log_dir = "Logs_MoE_Third_Architecture"
    os.makedirs(log_dir, exist_ok=True)
    
    val_loader = FastTensorDataLoader(X_val, y_val, batch_size=8192, shuffle=False)
    test_loader = FastTensorDataLoader(X_test, y_test, batch_size=8192, shuffle=False)
    
    moe_model.eval()
    
    # Evaluación en validación
    print("Evaluando en validación...")
    y_true_val, y_pred_val = [], []
    with torch.no_grad():
        for batch_X, batch_y in val_loader:
            with torch.amp.autocast(device.type):
                outputs = moe_model(batch_X)
            y_true_val.append(batch_y.cpu().numpy())
            y_pred_val.append(outputs.argmax(1).cpu().numpy())
    
    y_true_val = np.concatenate(y_true_val)
    y_pred_val = np.concatenate(y_pred_val)
    
    # Reporte de validación
    reporte_val = classification_report(
        y_true_val, y_pred_val,
        target_names=[str(c) for c in class_names],
        digits=4
    )
    with open(f"{log_dir}/Reporte_Val_{method_name}.txt", "w") as f:
        f.write(reporte_val)
    
    save_confusion_matrix(y_true_val, y_pred_val,f"MoE_CM_{method_name}_Val", phase="Validation")
    
    # Evaluación en test
    print("Evaluando en test...")
    y_true_test, y_pred_test = [], []
    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            with torch.amp.autocast(device.type):
                outputs = moe_model(batch_X)
            y_true_test.append(batch_y.cpu().numpy())
            y_pred_test.append(outputs.argmax(1).cpu().numpy())
    
    y_true_test = np.concatenate(y_true_test)
    y_pred_test = np.concatenate(y_pred_test)
    
    # Reporte de test
    reporte_test = classification_report(
        y_true_test, y_pred_test,
        target_names=[str(c) for c in class_names],
        digits=4
    )
    with open(f"{log_dir}/Reporte_Test_{method_name}.txt", "w") as f:
        f.write(reporte_test)
    
    save_confusion_matrix(y_true_test, y_pred_test, f"MoE_CM_{method_name}_Test", phase="Test")
    
    # Matriz de uso de expertos
    print("Generando matriz de uso de expertos...")
    expert_usage = save_expert_usage_matrix(moe_model, test_loader, method_name)
    
    # Métricas resumen
    f1_macro = f1_score(y_true_test, y_pred_test, average='macro')
    f1_weighted = f1_score(y_true_test, y_pred_test, average='weighted')
    accuracy = (y_true_test == y_pred_test).mean()
    
    print(f"\nResultados Finales {method_name}")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"F1 Macro: {f1_macro:.4f}")
    print(f"F1 Weighted: {f1_weighted:.4f}")
    
    resultados = {
        'accuracy': float(accuracy),
        'f1_macro': float(f1_macro),
        'f1_weighted': float(f1_weighted),
        'classification_report_test': reporte_test,
        'classification_report_val': reporte_val
    }
    
    with open(f"{log_dir}/Resultados_{method_name}.json", "w") as f:
        json.dump(resultados, f, indent=4)
    
    return resultados

In [ ]:

def pipeline_maestro_tabtransformer(metodo_oversampling,X_train_full, y_train_full, X_val, y_val_1d, X_test, y_test_1d, clases_por_experto, class_names, n_trials_expertos=100, n_trials_gating=100):
    
    input_dim_total = X_train_full.shape[1]
    
    # Fase 1: Pre-entrenamiento de expertos
    print("\nFase 1: Pre-entrenando expertos TabTransformer")
    lista_expertos = automatizar_fase1_expertos(
        metodo_oversampling=metodo_oversampling,
        X_val=X_val,
        y_val_1d=y_val_1d,
        clases_por_experto=clases_por_experto,
        n_trials=n_trials_expertos
    )
    
    # Fase 2: Entrenamiento del gating
    print("\nFase 2: Entrenando red de enrutamiento")
    moe_final = automatizar_fase2_gating(
        metodo_oversampling=metodo_oversampling,
        lista_expertos=lista_expertos,
        X_train=X_train_full,
        y_train=y_train_full,
        X_val=X_val,
        y_val=y_val_1d,
        input_dim=input_dim_total,
        n_trials=n_trials_gating
    )
    
    print("\nEvaluando modelo final")
    resultados = automatizar_evaluacion_moe(
        moe_model=moe_final,
        X_val=X_val,
        y_val=y_val_1d,
        X_test=X_test,
        y_test=y_test_1d,
        class_names=class_names,
        method_name=metodo_oversampling.upper()
    )
    
    print(f"\nPipeline {metodo_oversampling} completado")
    
    del moe_final, lista_expertos
    torch.cuda.empty_cache()
    
    return resultados

In [ ]:

optuna.logging.set_verbosity(optuna.logging.WARNING)
    
metodos = ['smote', 'smote_tomek', 'smote_enn']
    
todos_resultados = {}
    
for metodo in metodos:
    print(f"MoE con metodo {metodo.upper()}")
        
    try:
        resultados = pipeline_maestro_tabtransformer(
            metodo_oversampling=metodo,
            X_train_full=X_train_full,
            y_train_full=y_train_full,
            X_val=X_val,
            y_val_1d=y_val_1d,
            X_test=X_test,
            y_test_1d=y_test_1d,
            clases_por_experto=clases_por_experto,
            class_names=class_names,
            n_trials_expertos=100,
            n_trials_gating=100
        )
        todos_resultados[metodo] = resultados
            
    except Exception as e:
        print(f"Error con {metodo}: {e}")
    
print("\nPipeline completado exitosamente")